# 多端口校准

此示例演示如何使用 `MultiportSOLT` 校准类对具有三个或更多端口的多端口 S 参数测量进行校准。

## 设置

In [ ]:
import numpy as np

import skrf
from skrf.calibration import SOLT, MultiportCal, MultiportSOLT, UnknownThru, terminate_nport
from skrf.frequency import Frequency
from skrf.media import Coaxial
from skrf.network import connect, twoport_to_nport

skrf.stylely()

## 创建合成数据

首先，我们创建一个合成误差网络。

In [ ]:
freq = Frequency(1,100,100, unit='GHz')
nports = 3
# 1.0 mm coaxial media for calibration error boxes
coax = Coaxial(freq, z0_port=50, Dint=0.44e-3, Dout=1.0e-3, sigma=1e8)

# Create random error network
# First 'nports' ports will connect to an ideal VNA
# and the last 'nports' ports will connect to the DUT.
Z = coax.random(n_ports = 2*nports, name = 'Z')

# Zero leakage terms in the error network.
port_type = lambda n: 'VNA' if n < nports else 'DUT'  # noqa: E731
port_number = lambda n: n if n < nports else n - nports  # noqa: E731
for i in range(2*nports):
    for j in range(i+1, 2*nports):
        # No connection between different VNA/DUT ports.
        # No connection between VNA and DUT ports with different number.
        if port_type(i) == port_type(j) or port_number(i) != port_number(j):
            Z.s[:,i,j] = 0
            Z.s[:,j,i] = 0

# VNA switch terms. This is the termination impedance of each port
# when the source is not connected to that port.
gammas = []
for i in range(nports):
    gammas.append(coax.random(n_ports=1, name=f'gamma_{i}'))

用于通过误差网络测量网络的函数。简单地将误差网络连接到 DUT 网络，并添加端接阻抗。

In [ ]:
def measure(ntwk, Z, gammas, nports):
    out = terminate_nport(connect(Z, nports, ntwk, 0, num=nports), gammas)
    out.name = ntwk.name
    return out

接下来，创建校准标准。它们应该是多端口网络。在使用真实的 VNA 测量时，可能需要手动将 2 端口标准测量组合成 N 端口。

所需的标准是每个端口的 open、short 和 match，以及从第一个端口到所有其他端口的 thru 测量。

In [ ]:
o = coax.open(nports=nports, name='open')
s = coax.short(nports=nports, name='short')
m = coax.match(nports=nports, name='load')

thru = coax.thru(name='thru')

ideals = []
# nports-1 thrus from port 0 to all other ports.
for i in range(1, nports):
    thru_i = twoport_to_nport(thru, 0, i, nports)
    ideals.append(thru_i)

ideals.extend([o,s,m])
measured = [measure(k, Z, gammas, nports) for k in ideals]

用于测试校准的设备。一个简单的三路连接器。由于误差网络是随机生成的且具有非常高的反射，未校准网络的 S 参数无法识别。

In [ ]:
dut = coax.tee(name='dut')
dut_meas = measure(dut, Z, gammas, nports)
dut_meas.plot_s_db()

## 校准

创建 `MultiportSOLT` 校准类。第一个参数是类用于校准多端口的 2 端口校准方法。在这种情况下，我们使用简单的 SOLT 校准。其他可能性例如 `UnknownThru` 和 `LRRM`。请注意，不同的校准算法可能需要按特定顺序提供标准，并且可能需要额外的输入，如 `switch_terms`。有关所需输入，请参考 2 端口校准文档。

In [ ]:
cal = MultiportSOLT(method=SOLT, measured=measured, ideals=ideals)

校准 DUT 并绘制校准后的 S 参数。

In [ ]:
dut_cal = cal.apply_cal(dut_meas)
dut_cal.plot_s_db()

校正后的 DUT S 参数与原始 DUT S 参数之间的差异应该很小。

In [ ]:
np.max(np.abs(dut_cal.s - dut.s))

## MultiportCal

上面的 `MultiportSOLT` 类只能用于在两个端口之间只需要一个传输标准的 SOLT 风格 2 端口校准。多端口 TRL 需要在端口之间有 thru 和一个或多个线路，这不符合上述类接口。另外，如果由于某种原因应该对端口对使用不同的 2 端口校准方法，`MultiportSOLT` 类无法完成。对于这些情况，有一个更底层的 `MultiportCal`。

此校准需要端口对字典作为输入。'method' 键是 2 端口校准类，'measured' 是测量的网络。它们可以是 2 端口或 N 端口。其他键传递给 2 端口校准例程。在这种情况下，我们使用 `UnknownThru` 类，它需要额外的 `switch_term` 参数。请注意，2 端口开关项的顺序与多端口开关项的顺序相反。2 端口开关项按源连接的位置列出，但多端口开关项中未指定此点，它们按终止的端口列出。将标准和测量的网络按 2 端口校准所需的正确顺序列出也很重要。`UnknownThru` 假设 thru 应该列在最后，其他标准的顺序不重要。

In [ ]:
ideals_01 = ideals[2:] + [ideals[0]]
ideals_02 = ideals[2:] + [ideals[1]]
meas_01 = measured[2:] + [measured[0]]
meas_02 = measured[2:] + [measured[1]]

cal_dict = {(0, 1): {'method': UnknownThru, 'ideals': ideals_01,
                     'measured': meas_01, 'switch_terms':[gammas[1], gammas[0]]},
            (0, 2): {'method': UnknownThru, 'ideals': ideals_02,
                     'measured': meas_02, 'switch_terms':[gammas[2], gammas[0]]}}

cal2 = MultiportCal(cal_dict)

dut_cal2 = cal2.apply_cal(dut_meas)
dut_cal2.plot_s_db()